# Latihan 7 – Assignment
**Klasifikasi dataset `heart.csv` dengan k-NN dan Decision Tree**

Isi notebook, sesuai urutan soal:
1. Membaca dataset dan mengonversi data kategori menjadi numerik
2. Normalisasi: Min-Max (0-1), Z-Score, dan Sigmoidal
3. Klasifikasi: k-NN (k=1, 5, 9) dan Decision Tree
4. Validasi model: k-Fold (k=4, k=10) dan Leave-One-Out (LOO)
5. Menghitung error ratio dari setiap percobaan

> Letakkan `heart.csv` di folder yang sama dengan notebook ini.

## Soal 1 – Baca dataset dan konversi data kategori menjadi numerik

**Penjelasan.** Algoritma k-NN menghitung jarak antar data, dan Decision Tree dari scikit-learn memerlukan input angka. Jadi kolom bertipe teks (kategori) harus diubah menjadi angka terlebih dahulu.

Catatan: file `heart.csv` ini **tidak memiliki header** (baris pertama sudah berupa data), sehingga dibaca dengan `header=None` lalu kolom diberi nama sesuai dataset Statlog Heart.

Konversi yang dipakai adalah *label encoding* dengan mapping manual:
- Kategori biner (`sex`, `exercise_angina`, target) menjadi 0/1.
- Kategori multi-nilai (`chest_pain`, `rest_ecg`, `slope`, `thal`) diberi kode 0, 1, 2, ... sesuai urutan yang ditentukan di mapping.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, LeaveOneOut, cross_val_predict
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

cols = ["age", "sex", "chest_pain", "resting_bp", "cholesterol",
        "fasting_bs", "rest_ecg", "max_hr", "exercise_angina",
        "oldpeak", "slope", "ca", "thal", "target"]

df = pd.read_csv("heart.csv", header=None, names=cols)
print("Ukuran data:", df.shape)
df.head()

Ukuran data: (270, 14)


,age,sex,chest_pain,resting_bp,cholesterol,fasting_bs,rest_ecg,max_hr,exercise_angina,oldpeak,slope,ca,thal,target
0,70,male,asymptomatic,130,322,0,ventricular_hypertrophy,109,no,2.4,flat,3,normal,Presence
1,67,female,non_anginal_pain,115,564,0,ventricular_hypertrophy,160,no,1.6,flat,0,reversable_defect,Absence
2,57,male,atypical_angina,124,261,0,normal,141,no,0.3,upsloping,0,reversable_defect,Presence
3,64,male,asymptomatic,128,263,0,normal,105,yes,0.2,flat,1,reversable_defect,Absence
4,74,female,atypical_angina,120,269,0,ventricular_hypertrophy,121,yes,0.2,upsloping,1,normal,Absence


In [2]:
# Cek tipe data & nilai unik kolom kategori
print(df.dtypes, "\n")
for c in ["sex", "chest_pain", "rest_ecg", "exercise_angina", "slope", "thal", "target"]:
    print(c, "->", sorted(df[c].unique()))

age                  int64
sex                    str
chest_pain             str
resting_bp           int64
cholesterol          int64
fasting_bs           int64
rest_ecg               str
max_hr               int64
exercise_angina        str
oldpeak            float64
slope                  str
ca                   int64
thal                   str
target                 str
dtype: object 

sex -> ['female', 'male']
chest_pain -> ['asymptomatic', 'atypical_angina', 'non_anginal_pain', 'typical_angina']
rest_ecg -> ['normal', 'ventricular_hypertrophy', 'wave_abnormality']
exercise_angina -> ['no', 'yes']
slope -> ['downsloping', 'flat', 'upsloping']
thal -> ['fixed_defect', 'normal', 'reversable_defect']
target -> ['Absence', 'Presence']


In [3]:
mapping = {
    "sex":             {"female": 0, "male": 1},
    "chest_pain":      {"typical_angina": 0, "atypical_angina": 1,
                        "non_anginal_pain": 2, "asymptomatic": 3},
    "rest_ecg":        {"normal": 0, "wave_abnormality": 1,
                        "ventricular_hypertrophy": 2},
    "exercise_angina": {"no": 0, "yes": 1},
    "slope":           {"upsloping": 0, "flat": 1, "downsloping": 2},
    "thal":            {"normal": 0, "fixed_defect": 1, "reversable_defect": 2},
    "target":          {"Absence": 0, "Presence": 1},
}
for c, m in mapping.items():
    df[c] = df[c].map(m)

assert df.isna().sum().sum() == 0   # pastikan tidak ada nilai yang gagal dipetakan
print(df.dtypes)
df.head()

age                  int64
sex                  int64
chest_pain           int64
resting_bp           int64
cholesterol          int64
fasting_bs           int64
rest_ecg             int64
max_hr               int64
exercise_angina      int64
oldpeak            float64
slope                int64
ca                   int64
thal                 int64
target               int64
dtype: object


,age,sex,chest_pain,resting_bp,cholesterol,fasting_bs,rest_ecg,max_hr,exercise_angina,oldpeak,slope,ca,thal,target
0,70,1,3,130,322,0,2,109,0,2.4,1,3,0,1
1,67,0,2,115,564,0,2,160,0,1.6,1,0,2,0
2,57,1,1,124,261,0,0,141,0,0.3,0,0,2,1
3,64,1,3,128,263,0,0,105,1,0.2,1,1,2,0
4,74,0,1,120,269,0,2,121,1,0.2,0,1,0,0


In [4]:
X = df.drop(columns="target").values.astype(float)   # fitur
y = df["target"].values                              # label (0 = Absence, 1 = Presence)
print("X:", X.shape, "| y:", y.shape)
print("Distribusi kelas:", dict(zip(*np.unique(y, return_counts=True))))

X: (270, 13) | y: (270,)
Distribusi kelas: {np.int64(0): np.int64(150), np.int64(1): np.int64(120)}


## Soal 2 – Normalisasi: Min-Max (0-1), Z-Score, dan Sigmoidal

**Penjelasan.** Fitur pada dataset ini memiliki skala yang sangat berbeda (misalnya `cholesterol` ratusan, `sex` hanya 0/1). Pada k-NN, fitur berskala besar akan mendominasi perhitungan jarak. Normalisasi menyamakan skala fitur.

| Metode | Rumus | Hasil |
|---|---|---|
| Min-Max | $x' = \dfrac{x - x_{min}}{x_{max} - x_{min}}$ | rentang 0 sampai 1 |
| Z-Score | $x' = \dfrac{x - \mu}{\sigma}$ | rata-rata 0, simpangan baku 1 |
| Sigmoidal | $x' = \dfrac{1}{1 + e^{-z}}$, dengan $z$ = z-score | rentang (0, 1), outlier "ditekan" |

Sebagai pembanding, ditambahkan juga percobaan **tanpa normalisasi**.

In [5]:
def minmax(X):
    return (X - X.min(0)) / (X.max(0) - X.min(0))

def zscore(X):
    return (X - X.mean(0)) / X.std(0)

def sigmoidal(X):
    return 1.0 / (1.0 + np.exp(-zscore(X)))

datasets = {
    "Tanpa normalisasi": X,
    "Min-Max (0-1)":     minmax(X),
    "Z-Score":           zscore(X),
    "Sigmoidal":         sigmoidal(X),
}

# Ringkasan: rentang dan rata-rata hasil tiap normalisasi
ringkas = pd.DataFrame({
    nama: {"min": Xn.min(), "max": Xn.max(),
           "rata-rata": Xn.mean(), "std": Xn.std()}
    for nama, Xn in datasets.items()
}).T.round(3)
ringkas

,min,max,rata-rata,std
Tanpa normalisasi,0.000,564.000,45.584,79.201
Min-Max (0-1),0.000,1.000,0.404,0.383
Z-Score,-3.403,6.093,0.000,1.000
Sigmoidal,0.032,0.998,0.495,0.212


## Soal 3 – Klasifikasi dengan k-NN (k=1, k=5, k=9) dan Decision Tree

**Penjelasan.**
- **k-NN** memprediksi kelas sebuah data berdasarkan suara terbanyak dari *k* tetangga terdekat. *k* kecil (k=1) sensitif terhadap noise, *k* lebih besar lebih halus.
- **Decision Tree** membangun pohon keputusan dengan memecah data berdasarkan ambang nilai fitur. `random_state=42` dipakai agar hasil dapat direproduksi.

Model didefinisikan sebagai fungsi pembuat model supaya setiap fold mendapat model baru yang belum pernah dilatih.

In [6]:
models = {
    "k-NN (k=1)":    lambda: KNeighborsClassifier(n_neighbors=1),
    "k-NN (k=5)":    lambda: KNeighborsClassifier(n_neighbors=5),
    "k-NN (k=9)":    lambda: KNeighborsClassifier(n_neighbors=9),
    "Decision Tree": lambda: DecisionTreeClassifier(random_state=42),
}
list(models)

['k-NN (k=1)', 'k-NN (k=5)', 'k-NN (k=9)', 'Decision Tree']

## Soal 4 – Validasi model: k-Fold (k=4, k=10) dan LOO

**Penjelasan.** Validasi dipakai agar akurasi diukur pada data yang tidak ikut melatih model.
- **k-Fold:** data dibagi menjadi *k* bagian. Tiap bagian bergantian menjadi data uji, sisanya data latih. Dipakai k=4 dan k=10 (dengan `shuffle=True`, `random_state=42`).
- **Leave-One-Out (LOO):** kasus khusus k-Fold dengan k = jumlah data. Setiap iterasi hanya satu data yang diuji. Paling teliti, tetapi paling banyak membutuhkan komputasi (270 kali pelatihan).

In [7]:
validators = {
    "4-Fold":  KFold(n_splits=4,  shuffle=True, random_state=42),
    "10-Fold": KFold(n_splits=10, shuffle=True, random_state=42),
    "LOO":     LeaveOneOut(),
}
{k: v.get_n_splits(X) for k, v in validators.items()}   # jumlah iterasi tiap validasi

{'4-Fold': 4, '10-Fold': 10, 'LOO': 270}

## Soal 5 – Hitung error ratio dari setiap percobaan

**Penjelasan.** Error ratio adalah proporsi data yang salah diklasifikasikan:

$$\text{Error ratio} = \frac{\text{jumlah prediksi salah}}{\text{jumlah seluruh data}} = 1 - \text{akurasi}$$

Setiap data mendapat tepat satu prediksi saat menjadi data uji (`cross_val_predict`), lalu prediksi dari seluruh fold digabung dan dibandingkan dengan label asli.

Total percobaan: 4 normalisasi × 4 model × 3 validasi = **48 percobaan**.

In [8]:
rows = []
for norm_name, Xn in datasets.items():
    for model_name, make in models.items():
        for val_name, cv in validators.items():
            pred = cross_val_predict(make(), Xn, y, cv=cv)
            err = np.mean(pred != y)
            rows.append([norm_name, model_name, val_name, round(1 - err, 4), round(err, 4)])

res = pd.DataFrame(rows, columns=["Normalisasi", "Model", "Validasi", "Akurasi", "Error Ratio"])
res.to_csv("hasil_error_ratio.csv", index=False)
print("Jumlah percobaan:", len(res))
res.head(12)

Jumlah percobaan: 48


,Normalisasi,Model,Validasi,Akurasi,Error Ratio
0,Tanpa normalisasi,k-NN (k=1),4-Fold,0.5667,0.4333
1,Tanpa normalisasi,k-NN (k=1),10-Fold,0.5630,0.4370
2,Tanpa normalisasi,k-NN (k=1),LOO,0.5667,0.4333
3,Tanpa normalisasi,k-NN (k=5),4-Fold,0.6407,0.3593
4,Tanpa normalisasi,k-NN (k=5),10-Fold,0.6630,0.3370
5,Tanpa normalisasi,k-NN (k=5),LOO,0.6704,0.3296
6,Tanpa normalisasi,k-NN (k=9),4-Fold,0.6370,0.3630
7,Tanpa normalisasi,k-NN (k=9),10-Fold,0.6556,0.3444
8,Tanpa normalisasi,k-NN (k=9),LOO,0.6481,0.3519
9,Tanpa normalisasi,Decision Tree,4-Fold,0.7333,0.2667


### Tabel error ratio (baris = normalisasi & model, kolom = validasi)

In [9]:
pivot = res.pivot_table(index=["Normalisasi", "Model"], columns="Validasi",
                        values="Error Ratio")[["4-Fold", "10-Fold", "LOO"]]
pivot = pivot.reindex(list(datasets), level=0).reindex(list(models), level=1)
pivot.round(4)

Validasi                         4-Fold  10-Fold     LOO
Normalisasi       Model                                 
Tanpa normalisasi k-NN (k=1)     0.4333   0.4370  0.4333
                  k-NN (k=5)     0.3593   0.3370  0.3296
                  k-NN (k=9)     0.3630   0.3444  0.3519
                  Decision Tree  0.2667   0.2444  0.2778
Min-Max (0-1)     k-NN (k=1)     0.2370   0.2333  0.2444
                  k-NN (k=5)     0.1926   0.1963  0.1926
                  k-NN (k=9)     0.2000   0.1852  0.1778
                  Decision Tree  0.2667   0.2444  0.2741
Z-Score           k-NN (k=1)     0.2000   0.2259  0.2370
                  k-NN (k=5)     0.1630   0.1852  0.1741
                  k-NN (k=9)     0.1593   0.1630  0.1741
                  Decision Tree  0.2667   0.2444  0.2778
Sigmoidal         k-NN (k=1)     0.1963   0.2222  0.2333
                  k-NN (k=5)     0.1741   0.1741  0.1778
                  k-NN (k=9)     0.1704   0.1852  0.1852
                  Decision Tree  0.2704   0.2481  0.2778

In [10]:
# Kombinasi terbaik & terburuk
terbaik = res.loc[res["Error Ratio"].idxmin()]
terburuk = res.loc[res["Error Ratio"].idxmax()]
print("Terbaik :", terbaik.to_dict())
print("Terburuk:", terburuk.to_dict())

# Rata-rata error per normalisasi dan per model
print("\nRata-rata error per normalisasi:")
print(res.groupby("Normalisasi")["Error Ratio"].mean().round(4).sort_values())
print("\nRata-rata error per model:")
print(res.groupby("Model")["Error Ratio"].mean().round(4).sort_values())

Terbaik : {'Normalisasi': 'Z-Score', 'Model': 'k-NN (k=9)', 'Validasi': '4-Fold', 'Akurasi': 0.8407, 'Error Ratio': 0.1593}
Terburuk: {'Normalisasi': 'Tanpa normalisasi', 'Model': 'k-NN (k=1)', 'Validasi': '10-Fold', 'Akurasi': 0.563, 'Error Ratio': 0.437}

Rata-rata error per normalisasi:
Normalisasi
Z-Score              0.2059
Sigmoidal            0.2096
Min-Max (0-1)        0.2204
Tanpa normalisasi    0.3481
Name: Error Ratio, dtype: float64

Rata-rata error per model:
Model
k-NN (k=5)       0.2213
k-NN (k=9)       0.2216
Decision Tree    0.2633
k-NN (k=1)       0.2778
Name: Error Ratio, dtype: float64


## Kesimpulan

- **Normalisasi sangat berpengaruh pada k-NN**, karena k-NN berbasis jarak. Tanpa normalisasi, fitur berskala besar (misalnya `cholesterol`) mendominasi jarak sehingga error tinggi. Dengan Z-Score atau Sigmoidal, error turun jauh.
- **Decision Tree hampir tidak terpengaruh normalisasi**, karena pohon hanya membandingkan ambang nilai per fitur, bukan jarak antar fitur.
- **k=1 paling buruk** pada k-NN karena sensitif terhadap noise. k=5 dan k=9 lebih stabil.
- Ketiga metode validasi (4-Fold, 10-Fold, LOO) memberi hasil yang mirip. LOO paling stabil tetapi paling mahal secara komputasi.
- Kombinasi terbaik pada percobaan ini adalah **Z-Score + k-NN (k=9) dengan 4-Fold** (error ≈ 0.1593).

**Catatan metodologis:** normalisasi di sini diterapkan ke seluruh data sebelum validasi, sesuai bentuk soal. Pada penelitian sungguhan, parameter normalisasi sebaiknya dihitung hanya dari data latih di setiap fold agar tidak terjadi kebocoran informasi (*data leakage*).